[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saha23s/CD-MSC/blob/aaron%2Fpreprocessing/colab_dann.ipynb)

# CD-MSC — DANN Experiment

Trains with a **Gradient Reversal Layer** (Ganin et al. 2016) on the domain head.
The GRL pushes the backbone toward domain-invariant features, targeting a lower DSG.

**Before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Confirm GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU — check Runtime settings')

In [ ]:
# Clone repo if needed, then pull latest and install deps
import os
if not os.path.exists('/content/CD-MSC'):
    !git clone https://github.com/saha23s/CD-MSC.git /content/CD-MSC
%cd /content/CD-MSC
!git checkout aaron/preprocessing
!git pull origin aaron/preprocessing
!pip install -q -r requirements.txt

In [ ]:
# Mount Google Drive (needed to save outputs at the end)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Restore pre-extracted features from Drive
# (~5 GB pkl files — too large for the repo, saved here by colab_quickstart.ipynb)
import shutil, pathlib

src = pathlib.Path('/content/drive/MyDrive/CD-MSC-feature')
dst = pathlib.Path('Development_data/feature')
dst.mkdir(parents=True, exist_ok=True)

if src.exists():
    shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
    pkls = list(dst.glob('*.pkl'))
    print(f'Features restored: {len(pkls)} pkl files.')
else:
    print('ERROR: MyDrive/CD-MSC-feature not found on Drive.')
    print('Run colab_quickstart.ipynb first to extract and back up features.')

In [ ]:
# ── edit these ───────────────────────────────────────────────────────────────
DANN_ALPHA_MAX = 0.3   # 0.0 = off (reproduces baseline), 1.0 = full DANN
SEED           = 42
CDANN          = True  # True = conditional DANN (recommended), False = regular DANN
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Write config for this experiment
import json

with open('configs/default_experiment.json') as f:
    cfg = json.load(f)

cfg['seed'] = SEED
cfg['dann_alpha_max'] = DANN_ALPHA_MAX
cfg['cdann'] = CDANN

cdann_tag = '_cdann' if CDANN else ''
config_path = f'configs/dann_a{DANN_ALPHA_MAX}_seed{SEED}{cdann_tag}.json'
with open(config_path, 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'Config written: {config_path}')
print(f'  seed           = {SEED}')
print(f'  dann_alpha_max = {DANN_ALPHA_MAX}')
print(f'  cdann          = {CDANN}')
print()
print('Output dir will be:')
print(f'  outputs/MTRCNN_seed{SEED}_B64_E100_earlystop_min10_pati5_dann{DANN_ALPHA_MAX}{cdann_tag}/')

## Train

Expect ~20–40 min on T4. Early stopping monitors `val species_balanced_accuracy`.
The alpha (GRL weight) ramps from 0 → `DANN_ALPHA_MAX` using the Ganin schedule.

In [ ]:
!python train.py --config $config_path

In [ ]:
# Save outputs to Drive before session expires
import shutil
shutil.copytree('outputs', '/content/drive/MyDrive/CD-MSC-outputs', dirs_exist_ok=True)
print('Outputs saved to Google Drive.')

## Results

Compares the released 10-seed baseline against all DANN runs found in `outputs/`.

In [ ]:
import json, pathlib

def load_metrics(output_dir):
    p = pathlib.Path(output_dir) / 'best_model_eval' / 'test_metrics.json'
    return json.loads(p.read_text()) if p.exists() else None

rows = []

# Released 10-seed baseline mean (hard-coded from the paper)
rows.append({'run': 'Baseline 10-seed mean (released)', 'BA_seen': 0.8806, 'BA_unseen': 0.1751, 'DSG': 0.7055})

# Baseline seed 42 checkpoint (in repo)
m = load_metrics('outputs/MTRCNN_seed42_B64_E100_earlystop_min10_pati5')
if m:
    rows.append({'run': 'Baseline seed 42 (repo checkpoint)', 'BA_seen': m['BA_seen'], 'BA_unseen': m['BA_unseen'], 'DSG': m['DSG']})

# All DANN runs
for p in sorted(pathlib.Path('outputs').glob('*_dann*/best_model_eval/test_metrics.json')):
    m = json.loads(p.read_text())
    rows.append({'run': p.parts[-4], 'BA_seen': m['BA_seen'], 'BA_unseen': m['BA_unseen'], 'DSG': m['DSG']})

if not rows:
    print('No results found yet — run training first.')
else:
    w = 58
    print(f"{'Run':<{w}} {'BA_seen':>8} {'BA_unseen':>10} {'DSG':>8}")
    print('-' * (w + 30))
    for r in rows:
        marker = ' <-- DANN' if '_dann' in r['run'] else ''
        print(f"{r['run']:<{w}} {r['BA_seen']:>8.4f} {r['BA_unseen']:>10.4f} {r['DSG']:>8.4f}{marker}")